In [6]:
# ------------------------------
# 2-Stage Predictor
# ------------------------------

import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# ------------------------------
# Paths
# ------------------------------
STAGE1_MODEL = "C:/Users/Lenovo/Desktop/project_8D/models/stage1_group_classifier.h5"
STAGE1_CLASSES = "C:/Users/Lenovo/Desktop/project_8D/models/stage1_classes.json"
STAGE2_MODELS_DIR = "C:/Users/Lenovo/Desktop/project_8D/models"  # Stage2 models stored here

IMG_SIZE = 224

# ------------------------------
# Load Stage 1 model & classes
# ------------------------------
stage1_model = keras.models.load_model(STAGE1_MODEL)
with open(STAGE1_CLASSES, "r") as f:
    stage1_classes = json.load(f)
stage1_classes_inv = {v: k for k, v in stage1_classes.items()}

# ------------------------------
# Load Stage 2 models & classes
# ------------------------------
stage2_models = {}
stage2_classes = {}

for file in os.listdir(STAGE2_MODELS_DIR):
    if file.endswith("_model.h5"):
        group_name = file.replace("_model.h5", "")
        model_path = os.path.join(STAGE2_MODELS_DIR, file)
        class_path = os.path.join(STAGE2_MODELS_DIR, f"{group_name}_classes.json")
        stage2_models[group_name] = keras.models.load_model(model_path)
        with open(class_path, "r") as f:
            stage2_classes[group_name] = json.load(f)

# ------------------------------
# Prediction function
# ------------------------------
def predict_skin_disease(img_path):
    # Load and preprocess image
    img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Stage 1: Predict group
    stage1_pred = stage1_model.predict(img_array)
    group_idx = np.argmax(stage1_pred)
    group_name = stage1_classes_inv[group_idx]
    group_conf = stage1_pred[0][group_idx]

    # Stage 2: Predict specific disease
    stage2_model = stage2_models[group_name]
    stage2_pred = stage2_model.predict(img_array)
    disease_idx = np.argmax(stage2_pred)
    disease_name = list(stage2_classes[group_name].keys())[disease_idx]
    disease_conf = stage2_pred[0][disease_idx]

    return {
        "Predicted Group": group_name,
        "Group Confidence": float(group_conf),
        "Predicted Disease": disease_name,
        "Disease Confidence": float(disease_conf)
    }

# ------------------------------
# Example usage
# ------------------------------
if __name__ == "__main__":
    img_path = "C:/Users/Lenovo/Desktop/project_8D/images.jpg"
    result = predict_skin_disease(img_path)
    print(result)


1/1 [==============================] - 0s 462ms/step
{'Predicted Group': 'Pox', 'Group Confidence': 0.8948566317558289, 'Predicted Disease': 'Chickenpox', 'Disease Confidence': 0.8647732138633728}


In [1]:
# ------------------------------
# Fully Working Gradio GUI for 2-Stage Skin Disease Classifier
# ------------------------------

import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import gradio as gr
import matplotlib.pyplot as plt
from PIL import Image
import io

# ------------------------------
# Paths & Settings
# ------------------------------
STAGE1_MODEL = "C:/Users/Lenovo/Desktop/project_8D/models/stage1_group_classifier.h5"
STAGE1_CLASSES = "C:/Users/Lenovo/Desktop/project_8D/models/stage1_classes.json"
STAGE2_MODELS_DIR = "C:/Users/Lenovo/Desktop/project_8D/models"

IMG_SIZE = 224

# ------------------------------
# Load Stage 1 model & classes
# ------------------------------
stage1_model = keras.models.load_model(STAGE1_MODEL)
with open(STAGE1_CLASSES, "r") as f:
    stage1_classes = json.load(f)
stage1_classes_inv = {v: k for k, v in stage1_classes.items()}

# ------------------------------
# Load Stage 2 models & classes
# ------------------------------
stage2_models = {}
stage2_classes = {}

for file in os.listdir(STAGE2_MODELS_DIR):
    if file.endswith("_model.h5"):
        group_name = file.replace("_model.h5", "")
        model_path = os.path.join(STAGE2_MODELS_DIR, file)
        class_path = os.path.join(STAGE2_MODELS_DIR, f"{group_name}_classes.json")
        stage2_models[group_name] = keras.models.load_model(model_path)
        with open(class_path, "r") as f:
            stage2_classes[group_name] = json.load(f)

# ------------------------------
# Prediction function
# ------------------------------
def predict_skin_disease(img_path):
    img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Stage 1: Group prediction
    stage1_pred = stage1_model.predict(img_array, verbose=0)
    group_idx = np.argmax(stage1_pred)
    group_name = stage1_classes_inv[group_idx]
    group_conf = stage1_pred[0][group_idx]

    # Stage 2: Disease prediction
    stage2_model = stage2_models[group_name]
    stage2_pred = stage2_model.predict(img_array, verbose=0)
    disease_idx = np.argmax(stage2_pred)
    disease_name = list(stage2_classes[group_name].keys())[disease_idx]
    disease_conf = stage2_pred[0][disease_idx]

    # Full probability arrays for plotting
    return {
        "Predicted Group": group_name,
        "Group Confidence": float(group_conf),
        "Predicted Disease": disease_name,
        "Disease Confidence": float(disease_conf),
        "Stage1_Probs": stage1_pred[0],
        "Stage2_Probs": stage2_pred[0],
        "Stage2_Labels": list(stage2_classes[group_name].keys())
    }

# ------------------------------
# Function to generate bar plots
# ------------------------------
def plot_probabilities(labels, probs, title):
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.barh(labels, probs * 100)
    ax.set_xlim(0, 100)
    ax.set_xlabel("Confidence (%)")
    ax.set_title(title)
    for i, v in enumerate(probs * 100):
        ax.text(v + 1, i, f"{v:.2f}%", color='black', va='center')
    plt.tight_layout()
    
    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)
    plt.close(fig)
    return Image.open(buf)

# ------------------------------
# Gradio wrapper function
# ------------------------------
def gradio_predict(image):
    result = predict_skin_disease(image)

    # Text output
    output_text = f"""
Predicted Group: {result['Predicted Group']} ({result['Group Confidence']*100:.2f}%)
Predicted Disease: {result['Predicted Disease']} ({result['Disease Confidence']*100:.2f}%)
"""

    # Bar plots
    fig_group = plot_probabilities(
        labels=list(stage1_classes.keys()),
        probs=result['Stage1_Probs'],
        title="Group Prediction Confidence"
    )

    fig_disease = plot_probabilities(
        labels=result['Stage2_Labels'],
        probs=result['Stage2_Probs'],
        title=f"{result['Predicted Group']} Disease Confidence"
    )

    return output_text, fig_group, fig_disease

# ------------------------------
# Gradio Interface
# ------------------------------
demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Image(type="filepath", label="Upload Skin Image"),
    outputs=[
        gr.Textbox(label="Prediction"),
        gr.Image(type="pil", label="Group Confidence"),
        gr.Image(type="pil", label="Disease Confidence")
    ],
    title="DermaDiagnosis: Two-Stage Skin Disease Classifier",
    description="Upload a skin image. The system predicts the disease group and specific disease, with confidence scores and bar charts."
)

if __name__ == "__main__":
    demo.launch()


c:\Users\Lenovo\Desktop\project_8D\tfenv310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Lenovo\Desktop\project_8D\tfenv310\lib\site-packages\gradio_client\documentation.py:106: UserWarning: Could not get documentation group for <class 'gradio.mix.Parallel'>: No known documentation group for module 'gradio.mix'
  warnings.warn(f"Could not get documentation group for {cls}: {exc}")
c:\Users\Lenovo\Desktop\project_8D\tfenv310\lib\site-packages\gradio_client\documentation.py:106: UserWarning: Could not get documentation group for <class 'gradio.mix.Series'>: No known documentation group for module 'gradio.mix'
  warnings.warn(f"Could not get documentation group for {cls}: {exc}")




Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


IMPORTANT: You are using gradio version 3.43.0, however version 4.44.1 is available, please upgrade.
--------


predictor code for multimodels

In [3]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# ------------------------------
# Paths
# ------------------------------
MODEL_PATHS = {
    "CNN": "C:/Users/Lenovo/Desktop/project_8D/Algos/CNN_stage1/CNN_stage1.h5",
    "MobileNetV2": "C:/Users/Lenovo/Desktop/project_8D/Algos/MobileNetV2_stage1/MobileNetV2_stage1.h5",
    "EfficientNetB0": "C:/Users/Lenovo/Desktop/project_8D/Algos/EfficientNetB0_stage1/EfficientNetB0_stage1.h5",
    "ResNet50": "C:/Users/Lenovo/Desktop/project_8D/Algos/ResNet50_stage1/ResNet50_stage1.h5",
}

CLASS_JSON = "C:/Users/Lenovo/Desktop/project_8D/Algos/stage1_classes.json"

# ------------------------------
# Load Classes
# ------------------------------
with open(CLASS_JSON, "r") as f:
    class_indices = json.load(f)

# Reverse mapping: index -> class name
index_to_class = {v: k for k, v in class_indices.items()}

# ------------------------------
# Prediction Function
# ------------------------------
def predict_image(model_name, img_path, img_size=(224, 224)):
    if model_name not in MODEL_PATHS:
        raise ValueError(f"Model '{model_name}' not found!")

    # Load model
    model = tf.keras.models.load_model(MODEL_PATHS[model_name])

    # Load & preprocess image
    img = load_img(img_path, target_size=img_size)
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Predict
    preds = model.predict(img_array)
    pred_idx = np.argmax(preds)
    confidence = float(np.max(preds)) * 100

    result = {
        "model": model_name,
        "predicted_class": index_to_class[pred_idx],
        "confidence": f"{confidence:.2f}%"
    }
    return result

# ------------------------------
# Test
# ------------------------------
test_img = "C:/Users/Lenovo/Desktop/project_8D/images.jpg"
model_to_use = "MobileNetV2"
output = predict_image(model_to_use, test_img)
print(output)


1/1 [==============================] - 0s 480ms/step
{'model': 'MobileNetV2', 'predicted_class': 'Pox', 'confidence': '99.88%'}


In [ ]:
# gui_two_stage.py
# -----------------------------------------------
# Gradio GUI for Multi-Model 2-Stage Classifier
# -----------------------------------------------
import os
import glob
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import matplotlib.pyplot as plt
import gradio as gr
# ------------------------------
# Paths (adjust only if needed)
# ------------------------------
BASE_DIR = "C:/Users/Lenovo/Desktop/project_8D"
STAGE1_DIR = os.path.join(BASE_DIR, "Algos")
STAGE2_DIR = os.path.join(BASE_DIR, "Algos_2")

STAGE1_CLASSES_JSON = os.path.join(STAGE1_DIR, "stage1_classes.json")

STAGE1_MODEL_PATHS = {
    "CNN":            os.path.join(STAGE1_DIR, "CNN_stage1", "CNN_stage1.h5"),
    "MobileNetV2":    os.path.join(STAGE1_DIR, "MobileNetV2_stage1", "MobileNetV2_stage1.h5"),
    "EfficientNet":   os.path.join(STAGE1_DIR, "EfficientNet_stage1", "EfficientNet_stage1.h5"),
    "ResNet50":       os.path.join(STAGE1_DIR, "ResNet50_stage1", "ResNet50_stage1.h5"),
    "VGG19":          os.path.join(STAGE1_DIR, "VGG19_stage1", "VGG19_stage1.h5"),   # 🔹 Added
    "LSTM":           os.path.join(STAGE1_DIR, "LSTM_stage1", "LSTM_stage1.h5"),     # 🔹 Added
}

# 🔹 Must match filenames in Algos_2
STAGE2_BACKBONES = ["CNN", "MobileNetV2", "EfficientNet", "ResNet50", "VGG19", "LSTM"]

IMG_SIZE = 224


# ------------------------------
# Load Stage-1 class map once
# ------------------------------
with open(STAGE1_CLASSES_JSON, "r") as f:
    st1_name_to_idx = json.load(f)

st1_idx_to_name = {v: k for k, v in st1_name_to_idx.items()}
stage1_labels = [st1_idx_to_name[i] for i in range(len(st1_idx_to_name))]

# ------------------------------
# Cache for models
# ------------------------------
_stage1_cache = {}
_stage2_cache = {}

# ------------------------------
# Utilities
# ------------------------------
def load_and_preprocess(img_path, target=(IMG_SIZE, IMG_SIZE)):
    img = load_img(img_path, target_size=target)
    arr = img_to_array(img) / 255.0
    return np.expand_dims(arr, axis=0)

def get_stage1_model(name):
    if name not in STAGE1_MODEL_PATHS:
        raise ValueError(f"Unknown Stage-1 model: {name}")
    path = STAGE1_MODEL_PATHS[name]
    if not os.path.exists(path):
        raise FileNotFoundError(f"Stage-1 model not found: {path}")
    if name not in _stage1_cache:
        _stage1_cache[name] = keras.models.load_model(path, compile=False)
    return _stage1_cache[name]

def get_stage2_model(group_name, backbone=None):
    """
    Try exact file:  {group}_{backbone}_model.h5
    Fallback: first match {group}_*_model.h5
    Returns (model, classes_json_path, used_backbone)
    """
    used_backbone = backbone
    exact = None
    if backbone:
        exact = os.path.join(STAGE2_DIR, f"{group_name}_{backbone}_model.h5")

    classes_path = os.path.join(STAGE2_DIR, f"{group_name}_classes.json")
    if not os.path.exists(classes_path):
        return None, None, None

    if exact and os.path.exists(exact):
        key = (group_name, backbone)
        if key not in _stage2_cache:
            _stage2_cache[key] = keras.models.load_model(exact, compile=False)
        return _stage2_cache[key], classes_path, used_backbone

    matches = glob.glob(os.path.join(STAGE2_DIR, f"{group_name}_*_model.h5"))
    if matches:
        matches.sort()
        fallback_path = matches[0]
        parts = os.path.basename(fallback_path).split("_")
        if len(parts) >= 3:
            used_backbone = parts[1]
        key = (group_name, used_backbone)
        if key not in _stage2_cache:
            _stage2_cache[key] = keras.models.load_model(fallback_path, compile=False)
        return _stage2_cache[key], classes_path, used_backbone

    return None, None, None

def ordered_labels_from_name_to_idx(name_to_idx):
    inv = {v: k for k, v in name_to_idx.items()}
    return [inv[i] for i in range(len(inv))]

def make_barh(labels, probs, title):
    fig, ax = plt.subplots(figsize=(6, 3))
    probs = np.array(probs, dtype=float)
    ax.barh(labels, probs * 100.0)
    ax.set_xlim(0, 100)
    ax.set_xlabel("Confidence (%)")
    ax.set_title(title)
    for i, v in enumerate(probs * 100.0):
        ax.text(v + 1, i, f"{v:.2f}%", va="center")
    plt.tight_layout()
    return fig

# ------------------------------
# Core prediction
# ------------------------------
def run_two_stage(img_path, st1_choice, st2_backbone):
    # Stage 1
    m1 = get_stage1_model(st1_choice)
    x = load_and_preprocess(img_path)
    st1_probs = m1.predict(x, verbose=0)[0]
    st1_idx = int(np.argmax(st1_probs))
    group_name = st1_idx_to_name[st1_idx]
    group_conf = float(st1_probs[st1_idx])

    # Stage 2
    m2, classes_json, used_backbone = get_stage2_model(group_name, st2_backbone)
    if m2 is None:
        return {
            "group": group_name,
            "group_conf": group_conf,
            "group_probs": st1_probs,
            "group_labels": stage1_labels,
            "disease": f"No Stage 2 model found for '{group_name}'",
            "disease_conf": 0.0,
            "disease_probs": np.array([]),
            "disease_labels": [],
            "used_backbone": None,
        }

    with open(classes_json, "r") as f:
        st2_name_to_idx = json.load(f)
    st2_labels = ordered_labels_from_name_to_idx(st2_name_to_idx)

    y = m2.predict(x, verbose=0)[0]
    st2_idx = int(np.argmax(y))
    disease_name = st2_labels[st2_idx]
    disease_conf = float(y[st2_idx])

    return {
        "group": group_name,
        "group_conf": group_conf,
        "group_probs": st1_probs,
        "group_labels": stage1_labels,
        "disease": disease_name,
        "disease_conf": disease_conf,
        "disease_probs": y,
        "disease_labels": st2_labels,
        "used_backbone": used_backbone,
    }

# ------------------------------
# Gradio wrapper
# ------------------------------
def gradio_predict(img_path, st1_choice, st2_backbone):
    if img_path is None:
        return "Please upload an image.", "", "", None, None

    res = run_two_stage(img_path, st1_choice, st2_backbone)

    text_group = f"Group: {res['group']}  ({res['group_conf']*100:.2f}%)"
    if res["used_backbone"]:
        text_disease = f"Disease: {res['disease']}  ({res['disease_conf']*100:.2f}%)  • Stage-2 model: {res['used_backbone']}"
    else:
        text_disease = f"Disease: {res['disease']}"

    fig_group = make_barh(res["group_labels"], res["group_probs"], "Stage-1 Group Confidence")

    if len(res["disease_probs"]) == 0:
        fig_disease = plt.figure()
        plt.title("No Stage-2 model found")
    else:
        fig_disease = make_barh(res["disease_labels"], res["disease_probs"], f"Stage-2 ({res['used_backbone']}) Disease Confidence")

    return text_group, text_disease, f"Overall confidence (disease): {res['disease_conf']*100:.2f}%", fig_group, fig_disease

# ------------------------------
# Gradio UI
# ------------------------------
demo = gr.Interface(
    fn=gradio_predict,
    inputs=[
        gr.Image(type="filepath", label="Upload Skin Image"),
        gr.Dropdown(list(STAGE1_MODEL_PATHS.keys()), value="MobileNetV2", label="Stage-1 Model (Group)"),
        gr.Dropdown(STAGE2_BACKBONES, value="CNN", label="Stage-2 Backbone (Disease)"),
    ],
    outputs=[
        gr.Textbox(label="Group Prediction"),
        gr.Textbox(label="Disease Prediction"),
        gr.Textbox(label="Confidence"),
        gr.Plot(label="Group Probabilities"),
        gr.Plot(label="Disease Probabilities"),
    ],
    title="Skin Disease Classification (2-Stage, Multi-Model)",
    description="Stage-1 picks the group with your chosen backbone; Stage-2 picks the disease with your chosen backbone (or the first available model if not found).",
)

if __name__ == "__main__":
    demo.launch()


c:\Users\Lenovo\Desktop\project_8D\tfenv310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Lenovo\Desktop\project_8D\tfenv310\lib\site-packages\gradio_client\documentation.py:106: UserWarning: Could not get documentation group for <class 'gradio.mix.Parallel'>: No known documentation group for module 'gradio.mix'
  warnings.warn(f"Could not get documentation group for {cls}: {exc}")
c:\Users\Lenovo\Desktop\project_8D\tfenv310\lib\site-packages\gradio_client\documentation.py:106: UserWarning: Could not get documentation group for <class 'gradio.mix.Series'>: No known documentation group for module 'gradio.mix'
  warnings.warn(f"Could not get documentation group for {cls}: {exc}")


Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


IMPORTANT: You are using gradio version 3.43.0, however version 4.44.1 is available, please upgrade.
--------


In [1]:
import os

# Example values (aap apne code me stage1 aur user select se milenge)
group_name = "inflamatory"
backbone = "CNN"  # try "EfficientNet", "MobileNetV2", "ResNet50"

MODEL_DIR = "C:/Users/Lenovo/Desktop/project_8D/Algos_2"

# Debug prints
print(f"Looking for Stage-2 model for group '{group_name}' backbone '{backbone}'")

exact_model = os.path.join(MODEL_DIR, f"{group_name}_{backbone}_model.h5")
classes_path = os.path.join(MODEL_DIR, f"{group_name}_classes.json")

print("Exact model path:", exact_model)
print("Classes path:", classes_path)

# Check if files exist
print("Model exists?", os.path.exists(exact_model))
print("Classes exist?", os.path.exists(classes_path))


Looking for Stage-2 model for group 'inflamatory' backbone 'CNN'
Exact model path: C:/Users/Lenovo/Desktop/project_8D/Algos_2\inflamatory_CNN_model.h5
Classes path: C:/Users/Lenovo/Desktop/project_8D/Algos_2\inflamatory_classes.json
Model exists? True
Classes exist? True
